In [4]:
import os
import re
from glob import glob

import numpy as np
import pandas as pd
import uproot

from IPython.display import display

In [5]:
# ============================================================
# Configuration
# ============================================================

BASE_ROOT = "/scratch/elena/9Li/filtered_root"

# Isotope lifetimes [ms]
TAU_12B = 30.0
TAU_9LI = 257.0
TAU_16N = 7130.0

# Time regions after the end of the beam spill [ms]
REGION_1 = (20.0, 50.0)
REGION_2 = (50.0, 500.0)

# Detector / interaction assumptions used in the expectation
P_INT = 1.0
EFFICIENCY = 1.0

In [6]:
beam_groups = {
    "-340 MeV/c (UPW)": {
        "runs": [1846, 1848],
        "folder": "p_340",
    },

    "-260 MeV/c (UPW)": {
        "runs": [
            1928, 1930, 1932, 1934, 1935,
            1936, 1937, 1938, 1939, 1941
        ],
        "folder": "p_260",
    },

    "-270 MeV/c (Gd)": {
        "runs": [2407, 2408, 2409, 2432, 2438],
        "folder": "Gd/p_270",
    },

    "-350 MeV/c (Gd)": {
        "runs": [2374, 2379],
        "folder": "Gd/p_350",
    },

    "AmBe Bkg (Beam off)": {
        "runs": [2384],
        "folder": "AmBe_bkg",
    },
}

In [7]:
registered_spills = {
    1846: 269,
    1848: 254,

    1928: 440,
    1930: 267,
    1932: 539,
    1934: 370,
    1935: 379,
    1936: 317,
    1937: 441,
    1938: 256,
    1939: 410,
    1941: 516,

    2407: 350,
    2408: 474,
    2409: 569,
    2432: 214,
    2438: 1081,

    2374: 267,
    2379: 537,

    2384: None,
}

In [8]:
def get_root_file(run, sample="signal"):
    """
    Return the path to the filtered ROOT file for a given run.
    """

    for group_name, info in beam_groups.items():

        if run not in info["runs"]:
            continue

        folder = os.path.join(
            BASE_ROOT,
            info["folder"]
        )

        filename = (
            f"WCTE_merged_production_R{run}_{sample}.root"
        )

        path = os.path.join(folder, filename)

        return path if os.path.exists(path) else None

    return None

In [9]:
for group_name, info in beam_groups.items():

    print(f"\n{group_name}")

    for run in info["runs"]:

        sig = get_root_file(run, "signal")
        bkg = get_root_file(run, "bkg")

        print(
            f"Run {run}: "
            f"signal={'OK' if sig else 'MISSING'}, "
            f"bkg={'OK' if bkg else 'MISSING'}"
        )


-340 MeV/c (UPW)
Run 1846: signal=OK, bkg=OK
Run 1848: signal=OK, bkg=OK

-260 MeV/c (UPW)
Run 1928: signal=OK, bkg=OK
Run 1930: signal=OK, bkg=OK
Run 1932: signal=OK, bkg=OK
Run 1934: signal=OK, bkg=OK
Run 1935: signal=OK, bkg=OK
Run 1936: signal=OK, bkg=OK
Run 1937: signal=OK, bkg=OK
Run 1938: signal=OK, bkg=OK
Run 1939: signal=OK, bkg=OK
Run 1941: signal=OK, bkg=OK

-270 MeV/c (Gd)
Run 2407: signal=OK, bkg=OK
Run 2408: signal=OK, bkg=OK
Run 2409: signal=OK, bkg=OK
Run 2432: signal=OK, bkg=OK
Run 2438: signal=OK, bkg=OK

-350 MeV/c (Gd)
Run 2374: signal=OK, bkg=OK
Run 2379: signal=OK, bkg=OK

AmBe Bkg (Beam off)
Run 2384: signal=MISSING, bkg=OK


In [10]:
test_file = get_root_file(1846, "signal")

print(test_file)

with uproot.open(test_file) as f:

    print("\nROOT keys:")
    print(f.keys())

    tree = f["WCTEReadoutWindows"]

    print("\nBranches:")
    for branch in tree.keys():
        print(branch)

/scratch/elena/9Li/filtered_root/p_340/WCTE_merged_production_R1846_signal.root

ROOT keys:
['WCTEReadoutWindows;1', 'vme_analysis_scalar_results;1']

Branches:
window_time
spill_counter
start_counter
event_number
readout_number
nhit_mpmt_slot_ids
hit_mpmt_slot_ids
nhit_pmt_position_ids
hit_pmt_position_ids
nhit_pmt_charges
hit_pmt_charges
nhit_pmt_calibrated_times
hit_pmt_calibrated_times
window_data_quality_mask
vme_evt_quality_bitmask
vme_digi_issues_bitmask
T5_HasValidHit
T5_HasMultipleScintillatorsHit
T5_HasOutOfTimeWindow
T5_HasInTimeWindow
T5_particle_nr
vme_act_tagger
vme_act_eveto
vme_t0_time
vme_t1_time
vme_t4_time


In [11]:
tree = uproot.open(test_file)["WCTEReadoutWindows"]

print("\nNumber of entries:")
print(tree.num_entries)


Number of entries:
810237


In [14]:
def inspect_branch_values(run, sample="signal", n=10):

    path = get_root_file(run, sample)

    if path is None:
        print(f"No ROOT file found for run {run}")
        return

    tree = uproot.open(path)["WCTEReadoutWindows"]

    print(f"File: {path}")
    print(f"Entries: {tree.num_entries}")

    # Read only scalar branches
    branches = [
        "window_time",
        "spill_counter",
        "run_id",
        "event_number",
        "vme_t0_time",
        "vme_t1_time",
        "vme_t4_time",
        "vme_t5_time",
    ]

    # Keep only branches that actually exist
    available = set(tree.keys())
    branches = [b for b in branches if b in available]

    arrays = tree.arrays(
        branches,
        library="np",
        entry_stop=n
    )

    df = pd.DataFrame({
        b: arrays[b]
        for b in branches
    })

    display(df)


inspect_branch_values(1846, "signal", n=5)

File: /scratch/elena/9Li/filtered_root/p_340/WCTE_merged_production_R1846_signal.root
Entries: 810237


,window_time,spill_counter,event_number,vme_t0_time,vme_t1_time,vme_t4_time
0,7.224262e+09,0,0,-212.556248,-198.743752,-187.775009
1,7.237033e+09,0,1,-199.968754,-186.431255,-175.325005
2,7.239960e+09,0,2,-199.324997,-184.850001,-174.187508
3,7.244721e+09,0,3,-201.743752,-187.056252,-176.137497
4,7.246214e+09,0,4,-213.556236,-199.512489,-188.512497


In [15]:
# Inspect the scalar branches relevant for T5 / particle selection

path = get_root_file(1846, "signal")
tree = uproot.open(path)["WCTEReadoutWindows"]

branches = [
    "window_time",
    "spill_counter",
    "start_counter",
    "event_number",
    "T5_HasValidHit",
    "T5_HasMultipleScintillatorsHit",
    "T5_HasOutOfTimeWindow",
    "T5_HasInTimeWindow",
    "T5_particle_nr",
    "vme_act_tagger",
    "vme_act_eveto",
    "vme_t0_time",
    "vme_t1_time",
    "vme_t4_time",
]

arrays = tree.arrays(
    branches,
    library="np",
    entry_stop=20
)

df_test = pd.DataFrame({
    b: arrays[b]
    for b in branches
})

display(df_test)

,window_time,spill_counter,start_counter,event_number,T5_HasValidHit,T5_HasMultipleScintillatorsHit,T5_HasOutOfTimeWindow,T5_HasInTimeWindow,T5_particle_nr,vme_act_tagger,vme_act_eveto,vme_t0_time,vme_t1_time,vme_t4_time
0,7.224262e+09,0,903032722,0,True,False,True,False,1,47.006873,19.555506,-212.556248,-198.743752,-187.775009
1,7.237033e+09,0,904629114,1,True,False,True,True,2,57.283038,21.108817,-199.968754,-186.431255,-175.325005
2,7.239960e+09,0,904995003,2,True,False,False,True,1,48.103095,26.020993,-199.324997,-184.850001,-174.187508
3,7.244721e+09,0,905590123,3,True,False,True,True,2,22.947865,0.026817,-201.743752,-187.056252,-176.137497
4,7.246214e+09,0,905776738,4,True,False,True,False,2,53.841205,17.766111,-213.556236,-199.512489,-188.512497
5,7.246961e+09,0,905870145,5,True,False,True,False,3,NaN,NaN,-198.424995,-184.656246,-173.387497
6,7.249750e+09,0,906218707,6,True,False,True,False,7,NaN,NaN,-213.693752,-199.993752,-188.762497
7,7.249936e+09,0,906242006,7,True,False,True,False,1,NaN,NaN,-216.481251,-202.831242,-191.775009
8,7.250583e+09,0,906322896,8,True,False,True,True,2,27.654341,0.213725,-201.706249,-187.087498,-176.450005
9,7.251838e+09,0,906479778,9,True,True,True,True,12,NaN,NaN,-206.012505,-191.918751,-180.449997
